# NERC

Group 52, XB_0085 Text Mining

spaCy NER on the NER-test.tsv file. The dataset uses PER/ORG/LOC/MISC so we map spaCy's labels to those. Evaluation is token level plus seqeval entity level. Predictions and errors are written to data/ for the error analysis in the poster.

In [ ]:
from pathlib import Path
from seqeval.metrics import classification_report

import pandas as pd
import spacy


# spaCy uses different entity labels than the dataset
# This maps spaCy labels to the simpler course labels
SPACY_TO_COURSE_LABEL = {
    "PERSON": "PER",
    "ORG": "ORG",
    "GPE": "LOC",
    "LOC": "LOC",
    "FAC": "LOC",
    "NORP": "MISC",
    "LANGUAGE": "MISC",
    "PRODUCT": "MISC",
    "EVENT": "MISC",
    "WORK_OF_ART": "MISC",
    "LAW": "MISC",
}

In [ ]:
def read_ner_file(path):
    """
    Reads the NER test file

    Expected columns:
    sentence id, token id, token, BIO NER tag
    """
    df = pd.read_csv(path, sep="\t")

    expected_columns = {"sentence id", "token id", "token", "BIO NER tag"}
    missing = expected_columns - set(df.columns)

    if missing:
        raise ValueError(f"Missing expected columns: {missing}")

    return df


def make_sentences(df):
    """
    Groups the tokens back into sentences

    Returns a list of dictionaries:
    {
        "sentence_id": ...,
        "tokens": [...],
        "gold_tags": [...]
    }
    """
    sentences = []

    for sentence_id, group in df.groupby("sentence id", sort=False):
        group = group.sort_values("token id")

        tokens = group["token"].astype(str).tolist()
        gold_tags = group["BIO NER tag"].astype(str).tolist()

        sentences.append({
            "sentence_id": sentence_id,
            "tokens": tokens,
            "gold_tags": gold_tags,
        })

    return sentences


def predict_bio_tags(nlp, tokens):
    """
    Runs spaCy NER on a tokenized sentence and converts the output to BIO tags

    We use spaCy's Doc object so that spaCy keeps the original tokenization
    from the dataset
    """
    spaces = [True] * len(tokens)
    if spaces:
        spaces[-1] = False

    doc = spacy.tokens.Doc(nlp.vocab, words=tokens, spaces=spaces)
    doc = nlp(doc)

    predicted_tags = ["O"] * len(tokens)

    for ent in doc.ents:
        label = SPACY_TO_COURSE_LABEL.get(ent.label_)

        # Ignore labels that are not part of the course NER scheme,
        # such as DATE, TIME, MONEY, CARDINAL, etc.
        if label is None:
            continue

        for i, token in enumerate(ent):
            if i == 0:
                predicted_tags[token.i] = f"B-{label}"
            else:
                predicted_tags[token.i] = f"I-{label}"

    return predicted_tags

In [ ]:
def evaluate_token_level(gold_tags, predicted_tags):
    """
    Simple token-level evaluation

    This is not a perfect entity-level NER evaluation
    but it is useful for a first quantitative analysis
    """
    correct = 0
    total = 0

    label_counts = {}

    for gold, pred in zip(gold_tags, predicted_tags):
        total += 1

        if gold == pred:
            correct += 1

        if gold not in label_counts:
            label_counts[gold] = {"total": 0, "correct": 0}

        label_counts[gold]["total"] += 1

        if gold == pred:
            label_counts[gold]["correct"] += 1

    accuracy = correct / total if total > 0 else 0

    return accuracy, label_counts


def print_label_results(label_counts):
    """
    Prints accuracy for each gold label.
    """
    print("\nAccuracy by gold label:")

    for label, counts in sorted(label_counts.items()):
        total = counts["total"]
        correct = counts["correct"]
        accuracy = correct / total if total > 0 else 0

        print(f"{label:10s} {accuracy:.3f} ({correct}/{total})")


def save_predictions(sentences, output_path):
    """
    Saves token-level predictions to a TSV file
    """
    rows = []

    for sentence in sentences:
        sentence_id = sentence["sentence_id"]
        tokens = sentence["tokens"]
        gold_tags = sentence["gold_tags"]
        predicted_tags = sentence["predicted_tags"]

        for token_id, (token, gold, pred) in enumerate(
            zip(tokens, gold_tags, predicted_tags),
            start=1
        ):
            rows.append({
                "sentence id": sentence_id,
                "token id": token_id,
                "token": token,
                "gold tag": gold,
                "predicted tag": pred,
                "correct": gold == pred,
            })

    pd.DataFrame(rows).to_csv(output_path, sep="\t", index=False)


def save_errors(sentences, output_path):
    """
    Saves only the incorrect predictions.

    This file is useful for the qualitative error analysis in the poster.
    """
    rows = []

    for sentence in sentences:
        sentence_id = sentence["sentence_id"]
        tokens = sentence["tokens"]
        gold_tags = sentence["gold_tags"]
        predicted_tags = sentence["predicted_tags"]

        sentence_text = " ".join(tokens)

        for token_id, (token, gold, pred) in enumerate(
            zip(tokens, gold_tags, predicted_tags),
            start=1
        ):
            if gold != pred:
                rows.append({
                    "sentence id": sentence_id,
                    "token id": token_id,
                    "sentence": sentence_text,
                    "token": token,
                    "gold tag": gold,
                    "predicted tag": pred,
                })

    pd.DataFrame(rows).to_csv(output_path, sep="\t", index=False)

## Run

In [ ]:
input_path = "../data/NER-test.tsv"
model_name = "en_core_web_sm"
predictions_output = "../data/ner_predictions.tsv"
errors_output = "../data/ner_errors.tsv"

print("Loading data...")
df = read_ner_file(input_path)
sentences = make_sentences(df)

print(f"Loaded {len(sentences)} sentences.")
print(f"Loading spaCy model: {model_name}")

nlp = spacy.load(model_name)

all_gold_tags = []
all_predicted_tags = []

print("Running NERC...")

for sentence in sentences:
    predicted_tags = predict_bio_tags(nlp, sentence["tokens"])
    sentence["predicted_tags"] = predicted_tags

    all_gold_tags.extend(sentence["gold_tags"])
    all_predicted_tags.extend(predicted_tags)

## Evaluation

In [ ]:
accuracy, label_counts = evaluate_token_level(all_gold_tags, all_predicted_tags)

print("\nOverall token-level accuracy:")
print(f"{accuracy:.3f}")

print_label_results(label_counts)

print("\nEntity-level evaluation (seqeval):")
print(classification_report([all_gold_tags], [all_predicted_tags]))

In [ ]:
save_predictions(sentences, predictions_output)
save_errors(sentences, errors_output)

print("\nDone.")
print(f"Saved predictions to: {predictions_output}")
print(f"Saved errors to: {errors_output}")